In [1]:
import pandas as pd

In [2]:
prev_control_totals_xlsx = "_archive/_Source - ControlTotal_SE - 2022-08-31.xlsx"
gpi_employment_by_county_csv = "data/GPI-SE-Data-20251121/Employment_Projections_by_County_by_Industry_Vintage2025_for_WFRC.csv"

# Read in GPI employment data by county

In [3]:
gpi_employment_by_county_df = pd.read_csv(gpi_employment_by_county_csv)

gpi_employment_by_county_df.columns = gpi_employment_by_county_df.columns.str.replace(r"[^0-9A-Za-z]+", "_", regex=True).str.upper()

# Clean Industry names for merging, use lower case and remove non-alphabetic characters
gpi_employment_by_county_df["INDUSTRY"] = (
    gpi_employment_by_county_df["INDUSTRY"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z\s]+", "", regex=True)  # keep a–z and spaces
    .str.replace(r"\s+", " ", regex=True)       # collapse multiple spaces
    .str.strip()                                # remove leading/trailing spaces
)

# make sure only county level data is used
gpi_employment_by_county_df = gpi_employment_by_county_df[gpi_employment_by_county_df['AREATYPE']=='County']
gpi_employment_by_county_df.rename(columns={'AREA':'CO_NAME'}, inplace=True)
gpi_employment_by_county_df.drop(columns=['AREATYPE'], inplace=True)

# Standardize county names to uppercase for consistency
gpi_employment_by_county_df['CO_NAME'] = gpi_employment_by_county_df['CO_NAME'].str.upper()

gpi_employment_by_county_df

,YEAR,INDUSTRY,EMPCOUNT,CO_NAME,ISHISTORICAL
0,2023,accommodation and food services,517.000000,BEAVER,1
1,2024,accommodation and food services,481.000000,BEAVER,1
2,2025,accommodation and food services,483.522836,BEAVER,0
3,2026,accommodation and food services,496.285253,BEAVER,0
4,2027,accommodation and food services,497.151040,BEAVER,0
...,...,...,...,...,...
29923,2061,wholesale trade,5944.337619,WEBER,0
29924,2062,wholesale trade,5977.005576,WEBER,0
29925,2063,wholesale trade,6010.644866,WEBER,0
29926,2064,wholesale trade,6043.334823,WEBER,0


# Get employment lookup from previous control total file 

In [4]:
# open excel file and import worksheet ="Lookup" and range AB4:AL30 with header row
employment_classification_df = pd.read_excel(
    prev_control_totals_xlsx,
    sheet_name="Lookup",
    header=0,         # Row 4 is the header → zero-based index 3
    usecols="AB:AL",  # Restrict to columns AB through AL
    skiprows=lambda x: x < 3 or x > 29  # Keep rows 4–30 only
)
# Add underscores
employment_classification_df.columns = employment_classification_df.columns.str.replace(r"[^0-9A-Za-z]+", "_", regex=True).str.upper()

# Clean Industry names for merging, use lower case and remove non-alphabetic characters
employment_classification_df["GPI_INDUSTRY"] = (
    employment_classification_df["GPI_INDUSTRY"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z\s]+", "", regex=True)  # keep a–z and spaces
    .str.replace(r"\s+", " ", regex=True)       # collapse multiple spaces
    .str.strip()                                # remove leading/trailing spaces
)

# remove "State and Local Government" and "total jobs" rows that aren't in GPI data
employment_classification_df = employment_classification_df[
    ~employment_classification_df["GPI_INDUSTRY"].isin(["state and local government", "total jobs"])
].reset_index(drop=True)

# recode "State Government" and "Local Government" to "GVED" and corresponding IDs
mask = employment_classification_df["GPI_INDUSTRY"].isin(["state government", "local government"])
employment_classification_df.loc[mask, ["TDM_SECTOR", "TDM_SECTOR_ID", "TDM_7_SECTOR", "TDM_7_SECTOR_ID"]] = [
    ["GVED", 6, "OTH", 3]
]

# export to processed data folder
employment_classification_df.to_csv("intermediate/employment_classification_lookup.csv", index=False)

employment_classification_df

,GPI_INDUSTRY,GPI_NAICS,GOPB_SECTOR_ID,GOPB_SECTOR,TDM_SECTOR_ID,TDM_SECTOR,TDM_7_SECTOR_ID,TDM_7_SECTOR,ACS_SECTOR_ID,ACS_INDUSTRY_SECTORS,DWS
0,forestry fishing and hunting,113-115,1,Natural Resources,9,AGRI,4,AGRI,1,"Agriculture, forestry, fishing and hunting, an...",113-114
1,mining,21,2,Mining,10,MING,5,MING,1,"Agriculture, forestry, fishing and hunting, an...",21
2,utilities,22,3,Utilities,4,WSLE,2,IND,6,"Transportation and warehousing, and utilities",22
3,construction,23,4,Construction,11,CONS,6,CONS,2,Construction,23
4,manufacturing,31-33,5,Manufacturing,3,MANU,2,IND,3,Manufacturing,31-33
5,wholesale trade,42,6,Wholesale Trade,4,WSLE,2,IND,4,Wholesale trade,42
6,retail trade,44-45,7,Retail Trade,1,RETL,1,RET,5,Retail trade,44-45
7,transportation and warehousing,48-49,8,Transportation & Warehousing,4,WSLE,2,IND,6,"Transportation and warehousing, and utilities",48-49
8,information,51,9,Information,5,OFFI,3,OTH,7,Information,51
9,finance and insurance,52,10,Finance & Insurance,8,OTHR,3,OTH,8,"Finance and insurance, and real estate and ren...",52


In [5]:
gpi_to_tdm_emp_category_df = employment_classification_df[["GPI_INDUSTRY", "TDM_SECTOR"]].drop_duplicates()
gpi_to_tdm_emp_category_df.rename(columns={"GPI_INDUSTRY": "INDUSTRY"}, inplace=True)

# Clean INDUSTRY names for merging, use lower case and remove non-alphabetic characters
gpi_to_tdm_emp_category_df["INDUSTRY"] = (
    gpi_to_tdm_emp_category_df["INDUSTRY"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z\s]+", "", regex=True)  # keep a–z and spaces
    .str.replace(r"\s+", " ", regex=True)       # collapse multiple spaces
    .str.strip()                                # remove leading/trailing spaces
)

gpi_to_tdm_emp_category_df

,INDUSTRY,TDM_SECTOR
0,forestry fishing and hunting,AGRI
1,mining,MING
2,utilities,WSLE
3,construction,CONS
4,manufacturing,MANU
5,wholesale trade,WSLE
6,retail trade,RETL
7,transportation and warehousing,WSLE
8,information,OFFI
9,finance and insurance,OTHR


# Join and aggregate by TDM categories

In [6]:
# check for any industries in GPI employment data not in the lookup
left_only = set(gpi_employment_by_county_df["INDUSTRY"]) - set(gpi_to_tdm_emp_category_df["INDUSTRY"])
if left_only==set():
    print("All industries in GPI employment data have a matching TDM category.")

All industries in GPI employment data have a matching TDM category.


In [7]:
# check for any industries in lookup not in GPI employment data
right_only = set(gpi_to_tdm_emp_category_df["INDUSTRY"]) - set(gpi_employment_by_county_df["INDUSTRY"])
if right_only==set():
    print("All industries in the lookup have matching GPI employment data.")

All industries in the lookup have matching GPI employment data.


In [8]:
gpi_data_with_tdm_codes_df = pd.merge(gpi_employment_by_county_df, gpi_to_tdm_emp_category_df, on="INDUSTRY", how="outer")
gpi_data_with_tdm_codes_df

,YEAR,INDUSTRY,EMPCOUNT,CO_NAME,ISHISTORICAL,TDM_SECTOR
0,2023,accommodation and food services,517.000000,BEAVER,1,FOOD
1,2024,accommodation and food services,481.000000,BEAVER,1,FOOD
2,2025,accommodation and food services,483.522836,BEAVER,0,FOOD
3,2026,accommodation and food services,496.285253,BEAVER,0,FOOD
4,2027,accommodation and food services,497.151040,BEAVER,0,FOOD
...,...,...,...,...,...,...
29923,2061,wholesale trade,5944.337619,WEBER,0,WSLE
29924,2062,wholesale trade,5977.005576,WEBER,0,WSLE
29925,2063,wholesale trade,6010.644866,WEBER,0,WSLE
29926,2064,wholesale trade,6043.334823,WEBER,0,WSLE


In [9]:
# check for any rows with missing data after the merge
null_rows_df = gpi_data_with_tdm_codes_df[gpi_data_with_tdm_codes_df.isna().any(axis=1)]

if null_rows_df.size == 0:
    print("No mismatched rows in the merge.")

No mismatched rows in the merge.


In [10]:
tdm_emp_data_df = gpi_data_with_tdm_codes_df.groupby(['CO_NAME','YEAR','TDM_SECTOR']).agg({'EMPCOUNT':'sum'}).reset_index()
tdm_emp_data_df['EMPCOUNT'] = tdm_emp_data_df['EMPCOUNT'].astype(int)
tdm_emp_data_df

,CO_NAME,YEAR,TDM_SECTOR,EMPCOUNT
0,BEAVER,2023,AGRI,738
1,BEAVER,2023,CONS,278
2,BEAVER,2023,FOOD,517
3,BEAVER,2023,GVED,810
4,BEAVER,2023,HLTH,160
...,...,...,...,...
13712,WEBER,2065,MING,169
13713,WEBER,2065,OFFI,14763
13714,WEBER,2065,OTHR,56307
13715,WEBER,2065,RETL,20376


In [11]:
# Check sums from original GPI data and aggregated TDM data
original_total_emp = gpi_employment_by_county_df['EMPCOUNT'].sum()
aggregated_total_emp = tdm_emp_data_df['EMPCOUNT'].sum()
print(f"Original total employment: {original_total_emp}")
print(f"Aggregated total employment: {aggregated_total_emp}") 

Original total employment: 131989760.4699922
Aggregated total employment: 131983101


In [12]:
_2023_df = tdm_emp_data_df[tdm_emp_data_df['YEAR']==2023]
_2023_df['EMPCOUNT'].sum()
# 2023 All jobs in 20250307 Dataset is 2,426,629

np.int64(2426616)

In [13]:
from _counties import get_counties_df
counties_df = get_counties_df()
counties_df

,CO_FIPS,CO_NAME
0,5,CACHE
1,13,DUCHESNE
2,11,DAVIS
3,27,MILLARD
4,51,WASATCH
5,3,BOX ELDER
6,57,WEBER
7,23,JUAB
8,39,SANPETE
9,53,WASHINGTON


In [14]:
# join tdm_emp_data_df with counties_df to get CO_FIPS
tdm_emp_data_with_fips_df = pd.merge(
    tdm_emp_data_df,
    counties_df[['CO_NAME', 'CO_FIPS']],
    on='CO_NAME'
)
tdm_emp_data_with_fips_df = tdm_emp_data_with_fips_df[['CO_FIPS', 'YEAR', 'TDM_SECTOR', 'EMPCOUNT', 'CO_NAME']]
tdm_emp_data_with_fips_df

,CO_FIPS,YEAR,TDM_SECTOR,EMPCOUNT,CO_NAME
0,1,2023,AGRI,738,BEAVER
1,1,2023,CONS,278,BEAVER
2,1,2023,FOOD,517,BEAVER
3,1,2023,GVED,810,BEAVER
4,1,2023,HLTH,160,BEAVER
...,...,...,...,...,...
13712,57,2065,MING,169,WEBER
13713,57,2065,OFFI,14763,WEBER
13714,57,2065,OTHR,56307,WEBER
13715,57,2065,RETL,20376,WEBER


In [15]:
# export to processed data folder
tdm_emp_data_with_fips_df.to_csv("intermediate/tdm_employment_by_county.csv", index=False)